# Monte Carlo Simulation for European and Binary Option Pricing

## CQF Exam 2

**Author:** Mao Yikai  
**Date:** 2026/4/25  
**Objective:** Implement and compare numerical schemes for derivative pricing using risk-neutral valuation

---

## 1. Introduction (25%)

### 1.1 Problem Statement

The pricing of derivative securities is a fundamental problem in quantitative finance. Under the risk-neutral measure Q, the value of a derivative with payoff $\Phi(S_T)$ expiring at time $T$ is given by:

$$V(S, t) = e^{-r(T-t)} \mathbb{E}^Q[\Phi(S_T)]$$

where:
- $S$ is the underlying asset price
- $r$ is the constant risk-free interest rate
- $\mathbb{E}^Q[\cdot]$ denotes expectation under the risk-neutral measure

For derivatives without closed-form solutions, Monte Carlo simulation is a useful approach. In this project, we implement 3 numerical schemes to simulate the underlying asset price dynamics, evaluate their accuracy, and compare variance reduction techniques.

### 1.2 Geometric Brownian Motion and Black-Scholes Framework

Under the risk-neutral measure, the stock price evolves according to:

$$dS_t = rS_t dt + \sigma S_t dW_t$$

where $\sigma$ is the volatility and $W_t$ is a standard Brownian motion. The solution to this SDE is:

$$S_T = S_0 \exp\left(\left(r - \frac{\sigma^2}{2}\right)(T-t) + \sigma \sqrt{T-t} Z\right)$$

where $Z \sim \mathcal{N}(0,1)$ is a standard normal random variable. For European call options, the closed-form Black-Scholes formula provides:

$$C = S_0 N(d_1) - E e^{-r(T-t)} N(d_2)$$

where $d_1 = \frac{\ln(S_0/E) + (r + \sigma^2/2)(T-t)}{\sigma\sqrt{T-t}}$ and $d_2 = d_1 - \sigma\sqrt{T-t}$, with $N(\cdot)$ being the cumulative standard normal distribution (Hull, 2018).

### 1.3 Numerical Discretization Schemes

To compute the expectation numerically, we discretize the SDE over a time grid $0 = t_0 < t_1 < \ldots < t_N = T$ with step size $\Delta t = T/N$.

#### 1.3.1 Euler-Maruyama Scheme

The Euler-Maruyama scheme (Maruyama, 1955) provides the simplest approximation by discretizing the drift and diffusion terms:

$$S_{n+1} = S_n + rS_n \Delta t + \sigma S_n \Delta W_n$$

where $\Delta W_n \sim \mathcal{N}(0, \Delta t)$.

#### 1.3.2 Milstein Scheme

The Milstein scheme (Milstein, 1974) improves upon Euler-Maruyama by adding a correction term:

$$S_{n+1} = S_n + rS_n \Delta t + \sigma S_n \Delta W_n + \frac{1}{2}\sigma^2 S_n ((\Delta W_n)^2 - \Delta t)$$

This extra term captures nonlinear effects in the volatility dynamics, providing better accuracy for strong convergence in path-dependent scenarios.

#### 1.3.3 Closed Form Solution

We use the exact solution from section 1.2 to generate reference option prices via Monte Carlo. This provides a benchmark to assess the accuracy of the discretized schemes.

#### 1.3.4 Convergence Classification

The 3 schemes exhibit different convergence behaviors. We distinguish between **weak** and **strong** convergence:

- **Weak Convergence:** Convergence of expectations, $E[S_n^{\text{scheme}}] \to E[S_n^{\text{exact}}]$, relevant for option pricing. 
  - Euler-Maruyama: $O(\Delta t)$ weak order
  - Milstein: $O(\Delta t^2)$ weak order
  
- **Strong Convergence:** Convergence of sample paths, $E[|S_n^{\text{scheme}} - S_n^{\text{exact}}|] \to 0$, relevant for path-dependent options.
  - Euler-Maruyama: $O(\Delta t^{0.5})$ strong order
  - Milstein: $O(\Delta t^{1.0})$ strong order

For European options, where only the terminal payoff matters, weak convergence dominates. In practice, Monte Carlo variance ($O(N^{-0.5})$) typically dominates weak discretization bias, making higher-order weak schemes less critical than sample size (Kloeden & Platen, 1992).

We will apply these schemes to price 2 types of options with different payoff structures:

### 1.4 Option Types and Payoffs

**European Call Option:**  
$$\Phi_{EU}(S_T) = \max(S_T - E, 0)$$

**Binary Call Option (Cash-or-Nothing):**  
$$\Phi_{Binary}(S_T) = \mathbb{1}_{S_T > E}$$

### 1.5 Variance Reduction: Antithetic Variates

The antithetic variates technique (Hammersley & Handscomb, 1964) reduces Monte Carlo variance through symmetry. For each path $Z$, we generate a paired path $-Z$ and average their payoffs:

$$\hat{V}_{AV} = \frac{1}{2M} \sum_{i=1}^{M} \left[ e^{-rT} \Phi(S^{(i)}_T) + e^{-rT} \Phi(S^{(-i)}_T) \right]$$

This typically reduces variance by 50-90% for smooth payoffs, improving estimation accuracy without additional computational cost proportional to simulation count (Glasserman, 2004).

---

## 2. Implementation

### 2.1 Setup and Parameter Definition

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

# Configure plotting style
plt.style.use("seaborn-v0_8-darkgrid")
np.random.seed(42)

In [ ]:
# Base case parameters
base_params = {
    "S0": 100.0,        # Initial stock price
    "E": 100.0,         # Strike price
    "T": 1.0,           # Time to expiry (years)
    "sigma": 0.20,      # Volatility (20% p.a.)
    "r": 0.05,          # Risk-free rate (5% p.a.)
}

print("=" * 30)
print("BASE CASE PARAMETERS")
print("=" * 30)
for key, value in base_params.items():
    if key == "sigma":
        print(f"{"σ":>8}: {value:>8.2%}")
    elif key == "r":
        print(f"{key:>8}: {value:>8.2%}")
    else:
        print(f"{key:>8}: {value:>8.2f}")

BASE CASE PARAMETERS
      S0:   100.00
       E:   100.00
       T:     1.00
       σ:   20.00%
       r:    5.00%


### 2.2 Numerical Scheme Implementations

In [19]:
def euler_maruyama(S0: float, r: float, sigma: float, T: float, 
                   n_sims: int, n_steps: int) -> np.ndarray:
    """
    Simulate stock price paths using Euler-Maruyama discretization.
    
    Parameters:
        S0 (float)
        r (float)
        sigma (float): Volatility (annualized)
        T (float): Time to maturity (years)
        n_sims (int): Number of simulation paths
        n_steps (int): Number of time discretization steps
        
    Return:
        np.ndarray: Terminal stock prices of shape (n_sims,)
    """
    dt = T / n_steps
    dW = np.sqrt(dt) * np.random.randn(n_sims, n_steps)
    drift_coef = r * dt
    
    S = np.zeros((n_sims, n_steps + 1))
    S[:, 0] = S0
    
    for i in range(n_steps):
        diffusion_coef = sigma * dW[:, i]

        S[:, i + 1] = S[:, i] * (1 + drift_coef + diffusion_coef)
    
    return S[:, -1]

In [20]:
def milstein(S0: float, r: float, sigma: float, T: float, 
             n_sims: int, n_steps: int) -> np.ndarray:
    """
    Simulate stock price paths using Milstein discretization.
    
    Parameters:
        S0 (float)
        r (float)
        sigma (float): Volatility (annualized)
        T (float): Time to maturity (years)
        n_sims (int): Number of simulation paths
        n_steps (int): Number of time discretization steps
        
    Return:
        np.ndarray: Terminal stock prices of shape (n_sims,)
    """
    dt = T / n_steps
    dW = np.sqrt(dt) * np.random.randn(n_sims, n_steps)
    drift_coef = r * dt
    
    S = np.zeros((n_sims, n_steps + 1))
    S[:, 0] = S0
    
    for i in range(n_steps):
        diffusion_coef = sigma * dW[:, i]
        correction_coef = 0.5 * (sigma ** 2) * (dW[:, i] ** 2 - dt)
        
        S[:, i + 1] = S[:, i] * (1 + drift_coef + diffusion_coef + correction_coef)
    
    return S[:, -1]

In [21]:
def GBM_solution(S0: float, r: float, sigma: float, T: float, 
                       n_sims: int) -> np.ndarray:
    """
    Generate terminal stock prices using exact GBM solution.
    
    Parameters:
        S0 (float)
        r (float)
        sigma (float): Volatility (annualized)
        T (float): Time to maturity (years)
        n_sims (int): Number of simulation paths
        n_steps (int): Number of time discretization steps
        
    Return:
        np.ndarray: Terminal stock prices of shape (n_sims,)
    """
    Z = np.random.randn(n_sims)
    drift = (r - 0.5 * (sigma ** 2)) * T
    diffusion = sigma * np.sqrt(T) * Z
    
    return S0 * np.exp(drift + diffusion)

### 2.3 Payoff and Pricing Functions

In [22]:
def european_call_payoff(S: np.ndarray, E: float) -> np.ndarray:
    """
    Compute European call option payoff at maturity.
    
    Parameters:
        S (np.ndarray): Terminal stock prices
        E (float): Strike price
        
    Return:
        np.ndarray: Option payoffs
    """
    return np.maximum(S - E, 0)

In [23]:
def binary_call_payoff(S: np.ndarray, E: float) -> np.ndarray:
    """
    Compute binary call option payoff at maturity.
    
    Parameters:
        S (np.ndarray): Terminal stock prices
        E (float): Strike price
        
    Return:
        np.ndarray: Binary payoffs (0 or 1)
    """
    return (S > E).astype(int)

In [24]:
def monte_carlo_price(payoffs: np.ndarray, discount_factor: float) -> tuple:
    """
    Estimate option price from Monte Carlo payoffs.
    
    Parameters:
        payoffs (np.ndarray): Payoffs across all simulation paths
        discount_factor (float): exp(-r * T)
        
    Return:
        tuple: (price, std_error)
    """
    price = discount_factor * np.mean(payoffs)
    # Standard error of price estimate
    std_error = discount_factor * np.std(payoffs) / np.sqrt(len(payoffs))
    
    return price, std_error

In [25]:
def euler_maruyama_antithetic(S0: float, r: float, sigma: float, T: float, 
                              n_sims: int, n_steps: int) -> tuple:
    """
    Simulate paths using Euler-Maruyama with antithetic variates.
    For each random vector Z, generate both Z and -Z paths.
    
    Parameters:
        S0 (float)
        r (float)
        sigma (float): Volatility (annualized)
        T (float): Time to maturity (years)
        n_sims (int): Number of simulation paths
        n_steps (int): Number of time discretization steps
        
    Return:
        tuple: (S_positive, S_negative), and each has shape (n_sims,)
    """
    dt = T / n_steps
    dW = np.sqrt(dt) * np.random.randn(n_sims, n_steps)
    drift_coef = r * dt
    
    # Positive paths
    S_pos = np.zeros((n_sims, n_steps + 1))
    S_pos[:, 0] = S0
    for i in range(n_steps):
        diffusion_coef = sigma * dW[:, i]

        S_pos[:, i + 1] = S_pos[:, i] * (1 + drift_coef + diffusion_coef)
    
    # Negative (antithetic) paths
    S_neg = np.zeros((n_sims, n_steps + 1))
    S_neg[:, 0] = S0
    for i in range(n_steps):
        diffusion_coef = sigma * dW[:, i]

        S_neg[:, i + 1] = S_neg[:, i] * (1 + drift_coef - diffusion_coef)
    
    return S_pos[:, -1], S_neg[:, -1]

### 2.4 Black-Scholes Reference Formula

In [26]:
def black_scholes_call(S0: float, E: float, T: float, r: float, 
                       sigma: float) -> float:
    """
    Compute European call option price using Black-Scholes formula.
    
    
    Parameters:
        S0 (float)
        E (float)
        T (float): Time to maturity (years)
        r (float)
        sigma (float): Volatility (annualized)
    
    Return:
        float: European call option price
    """
    sqrt_T = np.sqrt(T)
    d1 = (np.log(S0 / E) + (r + 0.5 * (sigma ** 2)) * T) / (sigma * sqrt_T)
    d2 = d1 - sigma * sqrt_T
    
    call_price = S0 * norm.cdf(d1) - E * np.exp(-r * T) * norm.cdf(d2)
    
    return call_price

In [27]:
# Compute Black-Scholes reference price
bs_price = black_scholes_call(
    S0=base_params["S0"], 
    E=base_params["E"], 
    T=base_params["T"], 
    r=base_params["r"], 
    sigma=base_params["sigma"]
)

print(f"Black-Scholes European Call Price: ${bs_price:.4f}")

Black-Scholes European Call Price: $10.4506
